# **LOS/NLOS Classifier Model**

## **Project Setup**

### **Import libraries**
Import all basic libraries here. Additional libraries needed later in the notebook can be done at the code cells they are needed for.

In [ ]:
# import libraries
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    from scipy import stats

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

### **Load data**
Load all 7 indoor environment datasets into a single dataframe

In [ ]:
dataset_path = "../dataset/"  # Dataset directory

csv_files = [
    f for f in os.listdir(dataset_path) if f.endswith(".csv")
]  # Retrieve all csv files in the dataset folder

print(csv_files)  # print out all the csv file name in the 'dataset' folder

# Load and print out dataset values
dfs = []
for file in csv_files:
    print(f"Reading file: {file}")  # Print the file name being read
    df_temp = pd.read_csv(os.path.join(dataset_path, file))  # Read the CSV file
    # print(df_temp)  # Print the entire data of the current CSV file
    dfs.append(df_temp)  # Append the DataFrame to the list

df = pd.concat(
    dfs, ignore_index=True
)  # combining all datasets (csv) files into one DataFrame
print(df)

Reading the shape and columns of the dataframe provides information on all the data.

In [ ]:
# 'df' variable => combine all dataframe into one
print(
    "Number of rows and Column in dataset:", df.shape
)  # shape of the datasets (the total columns and rolls -> matrix[columns,rows])
print(
    "Number of rows in dataset:", df.columns
)  # total columns/attribute in the all datasets

print(
    "number of records in overall datasets:", len(df)
)  # Print the number of records in the all datasets

## **1 - Data Preparation (Data Cleaning & Data Preprocessing)**

#### **1.1 - Data Cleaning**

1. Check and Handle Missing Values

In [ ]:
# Calculate the missing values for each column
MissingValCount = df.isnull().sum()

# If there are missing values, drop them and create a new cleaned dataset
if MissingValCount.sum() > 0:
    df_clean = df.dropna()  # Drop rows with missing values
    print(
        f"Total missing values found and removed: {MissingValCount.sum()} rows dropped."
    )
else:
    df_clean = df.copy()  # If no missing values, just copy the original dataset
    print("No missing values found in datasets")

2. Display Dataset Shape Before and After Cleaning

In [ ]:
# Display the shape of the dataset before and after cleaning
print("Before cleaning, dataset rows and columns:", df.shape)
print("After cleaning, dataset rows and columns:", df_clean.shape)

# Check if there is a difference in rows or columns
if df.shape == df_clean.shape:
    print("The number of rows and columns are the same in both datasets.")
else:
    rows_diff = df.shape[0] - df_clean.shape[0]
    cols_diff = df.shape[1] - df_clean.shape[1]
    print(
        "The number of rows and columns are different between the original and cleaned datasets.",
        f"Rows difference: {rows_diff}, Columns difference: {cols_diff}",
    )

#### **1.2 -  Data Visualization and Target Variable Identification**


In [ ]:
# import libraries
try:
    import matplotlib.pyplot as plt
    import seaborn as sns

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

##### **1.2.1 - Target Variable: NLOS**

**Count Occurrences of Values**

In [ ]:
# count the total in NLOS attribute in the datasets
count_NLOS = df_clean["NLOS"].value_counts()

print(count_NLOS)

**Data Vizualization**

The pie chart below illustrates the count for **NLOS** with labels 0.0 and 1.0. From the graph, we can observe that the distribution is perfectly balanced, with an equal count of 0.0 and 1.0 labels.

In [ ]:
plt.figure(figsize=(6, 10))
count_NLOS.plot.pie(autopct="%1.1f%%", colors=["#3498db", "#e74c3c"], startangle=90)
plt.title("Distribution of NLOS")
plt.ylabel("")
plt.show()

##### **1.2.3 - Target Variable: FP_IDX**


**Identifying and Removing Outliers**

We be removing outliners Using IQR (Interquartile Range): 

1. Calculate IQR, Identify and Filter Outliers

In [ ]:
# Calculate Q1 (25th percentile) and Q3 (75th percentile) for FP_IDX
Q1 = df_clean["FP_IDX"].quantile(0.25)
Q3 = df_clean["FP_IDX"].quantile(0.75)

# Calculate the Interquartile Range (IQR)
IQR = Q3 - Q1

# Calculate lower and upper bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify outliers in FP_IDX based on the bounds
outliers = df_clean[
    (df_clean["FP_IDX"] < lower_bound) | (df_clean["FP_IDX"] > upper_bound)
]

# Print outliers
print("Outliers for 'FP_IDX':\n", outliers)

# Filter out outliers separately for LOS and NLOS
# For LOS (Line of Sight)
df_clean_filtered_LOS = df_clean[
    (df_clean["FP_IDX"] >= lower_bound)
    & (df_clean["FP_IDX"] <= upper_bound)
    & (df_clean["NLOS"] == 0.0)  # LOS condition
]

# For NLOS (Non-Line of Sight)
df_clean_filtered_NLOS = df_clean[
    (df_clean["FP_IDX"] >= lower_bound)
    & (df_clean["FP_IDX"] <= upper_bound)
    & (df_clean["NLOS"] == 1.0)  # NLOS condition
]

2. Combine Data, Clean Redundant Columns, and Rename

In [ ]:
# Combine both cleaned datasets (LOS and NLOS)
df_clean_filtered = pd.concat([df_clean_filtered_LOS, df_clean_filtered_NLOS])

# Remove redundant 'NLOS' column if it exists
if "category" in df_clean_filtered.columns and "NLOS" in df_clean_filtered.columns:
    df_clean_filtered = df_clean_filtered.drop(columns=["NLOS"])

# Rename 'NLOS' column to 'Signal_Path' if it exists
df_clean_filtered = (
    df_clean_filtered.rename(columns={"NLOS": "Signal_Path"})
    if "NLOS" in df_clean_filtered.columns
    else df_clean_filtered
)

After removing the outliners, next we will be filtering the dataset based on **FP_IDX** values within a specified range and adds a new **Category** column that labels the data as **"LOS"** or **"NLOS"** based on the NLOS values.

In [ ]:
df_clean_filtered = df_clean[
    (df_clean["FP_IDX"] >= lower_bound) & (df_clean["FP_IDX"] <= upper_bound)
].copy()
# Create a new column to indicate the category (LOS or NLOS)
df_clean_filtered.loc[:, "Category"] = df_clean_filtered["NLOS"].apply(
    lambda x: "LOS" if x == 1.0 else "NLOS"
)

print(df_clean_filtered)

**Data Vizualization**

The box plot below compares the distribution of **FP_IDX** values between **LOS** and **NLOS**. This helps in identifying differences in their statistical properties.

From the graph, we observe that **NLOS** has an outlier, indicating a potential anomaly, while **LOS** values are more evenly spread.

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x="Category", y="FP_IDX", data=df_clean_filtered)
plt.title("Boxplot of FP_IDX (LOS vs NLOS)")
plt.show()

##### **1.2.4 - Target Variable: FP_AMP1, FP_AMP2**, **FP_AMP3**

**Identifying and Removing Outliers**
We be removing outliners Using IQR (Interquartile Range): 

For FP_AMP1, 

In [ ]:
# Calculate the Range using IQR for FP_AMP1
Q1_FP_AMP1 = df_clean["FP_AMP1"].quantile(0.25)
Q3_FP_AMP1 = df_clean["FP_AMP1"].quantile(0.75)
IQR_FP_AMP1 = Q3_FP_AMP1 - Q1_FP_AMP1
lower_bound_FP_AMP1 = Q1_FP_AMP1 - 1.5 * IQR_FP_AMP1
upper_bound_FP_AMP1 = Q3_FP_AMP1 + 1.5 * IQR_FP_AMP1

# Identify outliers
outliers_FP_AMP1 = df_clean[
    (df_clean["FP_AMP1"] < lower_bound_FP_AMP1)
    | (df_clean["FP_AMP1"] > upper_bound_FP_AMP1)
]

# Filter out outliers based on the calculated bounds using IQR method
df_clean_filtered_FP_AMP1 = df_clean[
    (df_clean["FP_AMP1"] >= lower_bound_FP_AMP1)
    & (df_clean["FP_AMP1"] <= upper_bound_FP_AMP1)
]

# Print outliers
print("Outliers for 'FP_AMP1':\n", outliers_FP_AMP1)

For FP_AMP2 ,

In [ ]:
# Calculate the Range using IQR for FP_AMP2
Q1_FP_AMP2 = df_clean["FP_AMP2"].quantile(0.25)
Q3_FP_AMP2 = df_clean["FP_AMP2"].quantile(0.75)
IQR_FP_AMP2 = Q3_FP_AMP2 - Q1_FP_AMP2
lower_bound_FP_AMP2 = Q1_FP_AMP2 - 1.5 * IQR_FP_AMP2
upper_bound_FP_AMP2 = Q1_FP_AMP2 + 1.5 * IQR_FP_AMP2

# Identify outliers
outliers_FP_AMP2 = df_clean[
    (df_clean["FP_AMP2"] < lower_bound_FP_AMP2)
    | (df_clean["FP_AMP2"] > upper_bound_FP_AMP2)
]

# Filter out outliers based on the calculated bounds using IQR method
df_clean_filtered_FP_AMP2 = df_clean[
    (df_clean["FP_AMP2"] >= lower_bound_FP_AMP2)
    & (df_clean["FP_AMP2"] <= upper_bound_FP_AMP2)
]

# Print outliers
print("Outliers for 'FP_AMP2':\n", outliers_FP_AMP2)

For FP_AMP3 ,

In [ ]:
# Calculate the Range using IQR for FP_AMP3
Q1_FP_AMP3 = df_clean["FP_AMP3"].quantile(0.25)
Q3_FP_AMP3 = df_clean["FP_AMP3"].quantile(0.75)
IQR_FP_AMP3 = Q3_FP_AMP3 - Q1_FP_AMP3
lower_bound_FP_AMP3 = Q1_FP_AMP3 - 1.5 * IQR_FP_AMP3
upper_bound_FP_AMP3 = Q1_FP_AMP3 + 1.5 * IQR_FP_AMP3

# Identify outliers
outliers_FP_AMP3 = df_clean[
    (df_clean["FP_AMP3"] < lower_bound_FP_AMP3)
    | (df_clean["FP_AMP3"] > upper_bound_FP_AMP3)
]

# Filter out outliers based on the calculated bounds using IQR method
df_clean_filtered_FP_AMP3 = df_clean[
    (df_clean["FP_AMP3"] >= lower_bound_FP_AMP3)
    & (df_clean["FP_AMP3"] <= upper_bound_FP_AMP3)
]

# Print outliers
print("Outliers for 'FP_IDX':\n", outliers_FP_AMP3)

After removing the outliers, we combine FP_AMP1, FP_AMP2, and FP_AMP3 into one DataFrame

In [ ]:
# Combining multiplte dataframe in to one DataFrame
df_filtered = pd.DataFrame(
    {
        "FP_AMP1": df_clean_filtered_FP_AMP1["FP_AMP1"],
        "FP_AMP2": df_clean_filtered_FP_AMP2["FP_AMP2"],
        "FP_AMP3": df_clean_filtered_FP_AMP3["FP_AMP3"],
    }
)

# Reshape the data for the violin plot
df_filtered_melted = df_filtered.melt(var_name="Feature", value_name="Value")

**Data Vizualization**

After combining all into one DataFrame, we will plot a **violin graph** to illustrate the spread and median of **FP_AMP1**, **FP_AMP2**, and **FP_AMP3**, as their ranges are similar.

From the graph, we can see that these features have similar ranges, spread, and median. However, the median and spread for **FP_AMP1** are slightly higher than those of **FP_AMP2** and **FP_AMP3**, suggesting that **FP_AMP1** has slightly more variation or a higher central tendency compared to the other two.

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(x="Feature", y="Value", data=df_filtered_melted)

plt.title("Distribution of FP_AMP1, FP_AMP2, and FP_AMP3 (Outliers Removed)")
plt.show()

##### **1.2.5 - Target Variable: STDEV_NOISE**


For **STDEV_NOISE**, we will plot a strip plot to gain a better understanding of the distribution. The strip plot below illustrates the distribution of **STDEV_NOISE** with labels 0.0 and 1.0.

From the graph, we can observe that the 0.0 label has a lower **STDEV_NOISE** value compared to 1.0, indicating that the noise is more consistent in **LOS** (Line-of-Sight) than in **NLOS** (Non-Line-of-Sight).

In [ ]:
sns.stripplot(data=df_clean, x="NLOS", y="STDEV_NOISE", jitter=True)
plt.title("Distribution of STDEV_NOISE for LOS and NLOS")
plt.xlabel("NLOS (LOS vs NLOS)")
plt.ylabel("STDEV_NOISE")
plt.show()

##### **1.2.6 - Target Variable: CIR_PWR**

**Identifying and Removing Outliers**

We be removing outliners Using IQR (Interquartile Range): 

1. Calculate IQR and Identify Outliers

In [ ]:
# Calculate Q1 (25th percentile) and Q3 (75th percentile) for CIR_PWR
Q1_CIR_PWR = df_clean["CIR_PWR"].quantile(0.25)
Q3_CIR_PWR = df_clean["CIR_PWR"].quantile(0.75)

# Calculate the Interquartile Range (IQR) for CIR_PWR
IQR_CIR_PWR = Q3_CIR_PWR - Q1_CIR_PWR

# Calculate lower and upper bounds for outliers
lower_bound_CIR_PWR = Q1_CIR_PWR - 1.5 * IQR_CIR_PWR
upper_bound_CIR_PWR = Q3_CIR_PWR + 1.5 * IQR_CIR_PWR

# Identify outliers in CIR_PWR based on the calculated bounds
outliers_CIR_PWR = df_clean[
    (df_clean["CIR_PWR"] < lower_bound_CIR_PWR)
    | (df_clean["CIR_PWR"] > upper_bound_CIR_PWR)
]

2. Filter and Clean Data

In [ ]:
# Filter out outliers based on the calculated bounds using IQR method
df_clean_filtered_CIR_PWR = df_clean[
    (df_clean["CIR_PWR"] >= lower_bound_CIR_PWR)
    & (df_clean["CIR_PWR"] <= upper_bound_CIR_PWR)
]

# Print outliers
print("Outliers for 'CIR_PWR':\n", outliers_CIR_PWR)

**Data Vizualization**

After removing the outliers, we will plot a histogram to better understand the distribution of **CIR_PWER** values. T

The histogram below illustrates the spread of values for **CIR_PWER** with labels 0.0 and 1.0. Based on this graph, we can see that the distributions for XLOS and NLOS statuses are similar for **CIR_PWER**, but **NLOS** has a wider spread of values.

In [ ]:
sns.histplot(data=df_clean_filtered_CIR_PWR, x="CIR_PWR", hue="NLOS", kde=True)
plt.title("CIR_PWR Distribution for LOS vs NLOS")
plt.xlabel("CIR_PWR")
plt.ylabel("Count")
plt.grid(True)
plt.show()

##### **1.2.7 - Target Variable: MAX_NOISE**

For **MAX_NOISE**, we will plot a violin plot to better understand the distribution and observe the variation in noise. The plot below illustrates the distribution of MAX_NOISE with labels 0.0 and 1.0.

From the graph, we can observe how the noise varies between LOS (Line-of-Sight) and NLOS (Non-Line-of-Sight), highlighting the differences in their noise patterns.

In [ ]:
df_clean_filtered = df_clean[
    (df_clean["MAX_NOISE"] >= 500) & (df_clean["MAX_NOISE"] < 1100)
]

sns.violinplot(data=df_clean_filtered, x="NLOS", y="MAX_NOISE")
plt.title("Distribution of MAX_NOISE for LOS and NLOS Classes")
plt.xlabel("NLOS (LOS vs NLOS)")
plt.ylabel("MAX_NOISE")
plt.grid(True)
plt.show()

##### **1.2.8 - Target Variable: RXPACC**

**Identifying and Removing Outliers**
We be removing outliners Using IQR (Interquartile Range): 

Identify Outliers Using IQR for RXPACC

In [ ]:
# Filter the data based on specific RXPACC values (between 500 and 1100)
df_clean_filtered_RXPACC = df_clean[
    (df_clean["RXPACC"] >= 500) & (df_clean["RXPACC"] < 1100)
]

print(df_clean_filtered_RXPACC)

Next, we will remove columns that are not necessary for plotting the graph. This will help streamline the dataset, ensuring that we only work with the relevant features needed for visualization.

In [ ]:
# List of columns to exclude
columns_to_exclude = [
    "FP_IDX",
    "FP_AMP1",
    "FP_AMP2",
    "FP_AMP3",
    "CIR_PWR",
    "MAX_NOISE",
]

# Remove the CIR_[Number] columns using regex
columns_to_exclude += [
    col for col in df_clean_filtered_RXPACC.columns if col.startswith("CIR")
]
df_clean_copy = df_clean_filtered_RXPACC.drop(columns=columns_to_exclude)


print(df_clean_copy)

**Data Vizualization**

For **RXPACC** , we will plot a **KDE plot** to compare the distribution of **RXPACC** for **LOS (0.0)** and **NLOS (1.0)** classes. The plot helps highlight the density of **RXPACC** values, making it easy to spot any differences.

From the graph, we can see that **NLOS** has a more concentrated distribution of **RXPACC** values, while **LOS** shows a wider spread.

In [ ]:
sns.kdeplot(
    data=df_clean_copy, x="RXPACC", hue="NLOS", fill=True, palette={0: "blue", 1: "red"}
)
plt.title("KDE Plot of RXPACC for LOS (0.0) and NLOS (1.0)")
plt.xlabel("RXPACC")
plt.ylabel("Density")
plt.grid(True)
plt.show()

##### **1.2.9 - Target Variable: FRAME_LEN**

**Count Occurrences of Values**

In [ ]:
count_framelen = df_clean["FRAME_LEN"].value_counts()
print(count_framelen)

**Visualizing the Data**

For **FRAME_LEN**, we will plot a bar chart showing the count distribution, which allows us to see how the values are spread across the dataset.

From this graph, we can observe that **39.0** appears the most frequently, followed by **27.0**, with **29.0** being very rare in the dataset.`

In [ ]:
plt.figure(figsize=(10, 6))
count_framelen.plot.bar(color=["#3498db", "#e74c3c", "#f39c12"])
plt.title("Distribution of FRAME_LEN")
plt.ylabel("Count")
plt.xlabel("FRAME_LEN")
plt.xticks(rotation=0)

plt.yscale("linear")
plt.ylim(0, count_framelen.max() + 1000)

plt.show()

##### **1.2.10 - Target Variable: PREAM_LEN**

**Count Occurrences of Values**

In [ ]:
count_PREAM_LEN = df_clean["PREAM_LEN"].value_counts()
print(count_PREAM_LEN)

**Data Vizualization**

For **PREAM_LEN**, we will plot a bar chart showing the count distribution, which allows us to see how the values are spread across the dataset.

From this graph, we can observe that **1024.0** is overwhelmingly more common than **1536.0**, with **1024.0** making up the vast majority of the dataset.

In [ ]:
plt.figure(figsize=(10, 6))
count_PREAM_LEN.plot.bar(color=["#3498db", "#e74c3c"])
plt.title("Distribution of PREAM_LEN")
plt.ylabel("Total Count")
plt.xticks(rotation=0)
plt.show()

##### **1.2.1.11 - Target Variable: PRFR and BITRATE**

**Count Occurrences of Values**

In [ ]:
# For 'PRFR' values
count_PRFR = df_clean["PRFR"].value_counts()

# For 'BITRATE' values
count_BITRATE = df_clean["BITRATE"].value_counts()

**Data Vizualization**

Combine and Plot the Distribution of 'PRFR' and 'BITRATE'

After counting the Occurense for PRFR and BITRATE, the graph will display side-by-side bars showing their total count occurrences. Since the columns have fixed categories, the graph will provide a clear comparison of how each category in PRFR and BITRATE is distributed.

In [ ]:
combined_df = pd.DataFrame(
    {"PRFR": count_PRFR, "BITRATE": count_BITRATE, "BITRATE": count_BITRATE}
).fillna(
    0
)  # combine

combined_df.plot(kind="bar", figsize=(10, 6), color=["#3498db", "#e74c3c"])

plt.title("Combined Distribution of PRFR and BITRATE")
plt.ylabel("Total Count")
plt.xlabel("Categories")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

##### **1.2.1.12 - Target Variable: RANGE**

**Identifying and Removing Outliers**

We be removing outliners Using IQR (Interquartile Range): 

In [ ]:
Q1_RANGE = df_clean["RANGE"].quantile(0.25)
Q3_RANGE = df_clean["RANGE"].quantile(0.75)
IQR_RANGE = Q3_RANGE - Q1_RANGE
lower_bound_RANGE = Q1_RANGE - 1.5 * IQR_RANGE
upper_bound_RANGE = Q3_RANGE + 1.5 * IQR_RANGE

# Identify outliers
outliers_RANGE = df_clean[
    (df_clean["RANGE"] < lower_bound_RANGE) | (df_clean["RANGE"] > upper_bound_RANGE)
]

# Filter out outliers based on the calculated bounds using IQR method
df_clean_filtered_RANGE = df_clean[
    (df_clean["RANGE"] >= lower_bound_RANGE) & (df_clean["RANGE"] <= upper_bound_RANGE)
]

print(df_clean_filtered_RANGE)

**Data Vizualization**

For **RANGE**, we will plot a box plot to understand its distribution and identify outliers. 

From the graph, we can observe that **RANGE** has a wide spread with some potential outliers at the higher end.

In [ ]:
plt.figure(figsize=(10, 6))
plt.boxplot(
    df_clean_filtered_RANGE["RANGE"],
    vert=False,
    patch_artist=True,
    boxprops=dict(facecolor="lightblue", color="darkblue"),
    medianprops=dict(color="red"),
)
plt.title("Box Plot of RANGE")
plt.xlabel("RANGE")
plt.show()

### **1.3 - Feature Evaluation and Relationship Analysis**

After having a better understanding of the features in the dataset using visualization, next we need to identify which features have the most impact on predicting the target variable. This allows us to focus on the most relevant features, improving model performance and interpretability.

We be using different technique:


##### **1.3.1 - Correlation Matrix**

Firstly we be calculating the  **Correlation Matrix**, we will plot a heatmap to gain a better understanding of the relationships between different features. The heatmap below illustrates the correlation between each feature, with color intensity representing the strength of the relationship.

From the graph, we can observe that **RXPACC** has a strong positive correlation with **RANGE**, while **FP_AMP1**, **FP_AMP2**, and **FP_AMP3** show negative correlations with **RANGE**, indicating their potential influence on the target variable.

In [ ]:
features = [
    "RANGE",
    "FP_IDX",
    "FP_AMP1",
    "FP_AMP2",
    "FP_AMP3",
    "STDEV_NOISE",
    "CIR_PWR",
    "MAX_NOISE",
    "RXPACC",
    "FRAME_LEN",
    "PREAM_LEN",
]

correlation_matrix = df_clean[features].corr()
print(correlation_matrix)

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Matrix")
plt.show()

##### **1.3.2 - Mutual Information**

Secondly, we will calculate the **Mutual Information** to assess the relationship between each feature and the target variable **RANGE**. We will plot a bar chart to visualize the mutual information scores where higher scores indicate more significant features for prediction.

From the graph, we can observe that **RXPACC** has the highest mutual information score  making it the most important feature for predicting **RANGE**. Other features such as **MAX_NOISE** and **CIR_PWR** also have notable scores  suggesting their relevance. On the other hand, **FRAME_LEN** has a low score which indicating that it might have less influence on the target variable.

In [ ]:
# import libraries
try:
    from sklearn.feature_selection import mutual_info_regression

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

In [ ]:
# mutual information => (to check the relationship between the features)

# Features
features = [
    "FP_IDX",
    "FP_AMP1",
    "FP_AMP2",
    "FP_AMP3",
    "STDEV_NOISE",
    "CIR_PWR",
    "MAX_NOISE",
    "RXPACC",
    "FRAME_LEN",
    "PREAM_LEN",
]

# Define X as features set
X = df_clean[features]

# Define target variable
y = df_clean["RANGE"]

# Compute mutual information scores
mi_scores = mutual_info_regression(X, y)
mi_scores = pd.Series(mi_scores, index=features)

# Sort the scores in descending order and print
mi_scores = mi_scores.sort_values(ascending=False)
print("Mutual Information Scores (in descending order):")
print(mi_scores)

**Visualizing the Data**

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x=mi_scores.index, y=mi_scores.values)
plt.xlabel("Features")
plt.ylabel("Mutual Information Score")
plt.title("Feature Importance using Mutual Information")
plt.xticks(rotation=45)
plt.show()

##### **1.3.3 - Normalization**

Thirdly, we performed **Normalization** using **MinMaxScaler** to scale feature values between **0 and 1**. The histogram below shows the distribution of scaled features, ensuring they fall within the same range.

Most features are **well-scaled**, but some may show **skewness** or **concentration** at one end, indicating the need for further transformations. The histogram also highlights potential **outliers** that could impact model performance.

Normalization helps ensure **equal contribution** of all features, improving the **accuracy** and **stability** of the classifier.

In [ ]:
# import libraries

try:
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import MinMaxScaler

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

In [ ]:
X = df_clean.drop(columns=["NLOS"])

y = df_clean["NLOS"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalize features
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame
X_train = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test = pd.DataFrame(X_test_scaled, columns=X.columns)

print(X_train)

**Visualizing the Data**

In [ ]:
# Plot histograms for the first feature (RANGE) before and after scaling
plt.figure(figsize=(14, 6))

# Original data (before scaling)
plt.subplot(1, 2, 1)
plt.hist(X["RANGE"], bins=50, color="skyblue", edgecolor="black")
plt.title("Original RANGE Feature Distribution")
plt.xlabel("RANGE")
plt.ylabel("Frequency")

# Scaled data (after MinMaxScaler)
plt.subplot(1, 2, 2)
plt.hist(X_train["RANGE"], bins=50, color="salmon", edgecolor="black")
plt.title("Scaled RANGE Feature Distribution")
plt.xlabel("RANGE")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))

# Original data (before scaling)
plt.subplot(1, 2, 1)
sns.boxplot(x=X["RANGE"], color="skyblue")
plt.title("Original RANGE Feature Distribution")

# Scaled data (after MinMaxScaler)
plt.subplot(1, 2, 2)
sns.boxplot(x=X_train["RANGE"], color="salmon")
plt.title("Scaled RANGE Feature Distribution")

plt.tight_layout()
plt.show()

##### **1.3.4 - Principal Component Analysis (PCA)**

Lastly, we performed **PCA** to reduce the number of CIR features while retaining key information. PCA transforms the original features into principal components, which capture the most important patterns in the data. The first 14 components explain 95% of the variance, with the 15th capturing the remaining variance. This reduces the dataset's complexity while preserving essential details for further analysis and modeling.

## **2 - Data Mining & Splitting**


After feature evaluation, we divide the dataset into training and test sets to train the model and assess its performance on unseen data.

In [ ]:
# import libaries
try:
    from sklearn.model_selection import train_test_split
    import numpy as np

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

First, we split the dataset into training and testing sets to maintains the same structure but improves readability.

### **2.4.1 - Data Minning**

In [ ]:
# Remove CIR columns (PCA already reduced them)
df_clean = df_clean.loc[:, ~df_clean.columns.str.match(r"^CIR\d+$")]

# Drop unnecessary columns
columns_to_drop = ["FP_AMP2", "MAX_NOISE", "FRAME_LEN", "CH", "PRFR"]
df_clean = df_clean.drop(columns=columns_to_drop, errors="ignore")

# Find all FP_AMP columns and compute the average
fp_amp_cols = df_clean.filter(like="FP_AMP").columns

if len(fp_amp_cols) > 0:
    df_clean["FP_AMP_AVG"] = df_clean[fp_amp_cols].mean(axis=1)
    df_clean = df_clean.drop(columns=fp_amp_cols, errors="ignore")

# Features (excluding 'NLOS' since it's the target)
X = df_clean.drop(columns=["NLOS"], errors="ignore")

# Target variable
y = df_clean["NLOS"]


### **2.4.2 - Data Splitting**

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Print dataset shapes
print(f"Training Features Shape: {X_train.shape}")
print(f"Testing Features Shape: {X_test.shape}")
print(f"Training Labels Shape: {y_train.shape}")
print(f"Testing Labels Shape: {y_test.shape}")

## **3 - Model Training & Model Evaluation**

Once the dataset has been divided into training and testing sets, we will proceed with training the model using the training data and a selection of algorithms. Using multiple algorithms allows us to compare their performance and choose the one that best suits the problem at hand. Different algorithms may capture different patterns in the data, and leveraging a variety of approaches can enhance the model's ability to generalize and improve accuracy on unseen data.


In [ ]:
# import libaries
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import RandomizedSearchCV
    from sklearn.ensemble import RandomForestClassifier
    import itertools
    from sklearn.metrics import (
        roc_curve,
        roc_auc_score,
    )

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

#### **3.1.1 - Logistic Regression**

Firstly, we initializes a Logistic Regression model with specific hyperparameters and then trains the model on the scaled training data **(X_train_scaled)** to classify the target labels (y_train).

In [ ]:
# import libaries
try:
    from sklearn.metrics import (
        auc,
        accuracy_score,
        classification_report,
        confusion_matrix,
    )
    from sklearn.model_selection import learning_curve
    from sklearn.model_selection import GridSearchCV
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

In [ ]:
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

##### **3.1.1.1 - Hyperparameter Tuning** 


To further enhance the model’s performance, we perform Hyperparameter Tuning to find the best combination of parameters. The optimal hyperparameters were penalty: 'l2', C: 0.1, solver: 'saga', and max_iter: 200, improving the model’s accuracy and efficiency.

In [ ]:
param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10],  # Regularization strength
    "solver": ["liblinear", "lbfgs"],  # Solvers
    "max_iter": [1000, 2000],  # Max iterations
}

lr_model = LogisticRegression(random_state=42)

grid_search = GridSearchCV(lr_model, param_grid, cv=5, scoring="accuracy")
grid_search.fit(X_train_scaled, y_train)

best_lr_model = grid_search.best_estimator_
print(f"Best Parameters: {grid_search.best_params_}")

##### **3.1.1.2 - Model Training and Testing (Description to be modify)** 

**Training and Test ROC-AUC** 

In [ ]:
# Predict probabilities for training and test sets
y_train_pred_lr = best_lr_model.predict_proba(X_train_scaled)[:, 1]
y_test_pred_lr = best_lr_model.predict_proba(X_test_scaled)[:, 1]

# Compute ROC-AUC scores for both train and test sets
train_auc_lr = roc_auc_score(y_train, y_train_pred_lr)
test_auc_lr = roc_auc_score(y_test, y_test_pred_lr)

# Print AUC for train and test
print(
    f"Logistic Regression - Train ROC-AUC: {train_auc_lr:.4f}, Test ROC-AUC: {test_auc_lr:.4f}"
)

###### **3.1.3 - visualization graph** 

In [ ]:
fpr_train_lr, tpr_train_lr, _ = roc_curve(y_train, y_train_pred_lr)
fpr_test_lr, tpr_test_lr, _ = roc_curve(y_test, y_test_pred_lr)

plt.figure(figsize=(8, 6))
plt.plot(
    fpr_train_lr,
    tpr_train_lr,
    color="blue",
    label="Train ROC curve (AUC = {:.2f})".format(train_auc_lr),
)
plt.plot(
    fpr_test_lr,
    tpr_test_lr,
    color="red",
    label="Test ROC curve (AUC = {:.2f})".format(test_auc_lr),
)
plt.plot([0, 1], [0, 1], color="gray", linestyle="--")  # Random classifier line
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend(loc="lower right")
plt.grid(True)

##### **3.1.1.3  - Model Evaluation (to be modify)**

To further assess the model's performance, we evaluate key metrics, including **accuracy, confusion matrix, and classification report**. The **Logistic Regression** model achieved an **accuracy of 0.8560**, indicating strong overall performance across both the training and test data.  

The **Confusion Matrix** provides further insights into the model's classification performance. The model correctly identified **3870 True Negatives** and **3320 True Positives**, while misclassifying **382 False Positives** and **828 False Negatives**. These values suggest that the model performs well but misses some **NLOS** instances.  

The **Classification Report** provides additional details on precision, recall, and f1-score. The **precision** for **LOS (0.0)** is **0.82**, while for **NLOS (1.0)** it is **0.90**, indicating that the model is slightly better at predicting **NLOS** instances correctly. The **recall** for **LOS (0.0)** is **0.91**, demonstrating strong identification of **LOS**, whereas the **recall** for **NLOS (1.0)** is **0.80**, meaning some **NLOS** cases are misclassified. The **f1-scores** for both classes are balanced, with **LOS (0.0) at 0.86** and **NLOS (1.0) at 0.85**, reflecting the model’s overall effectiveness.  

Overall, the model shows strong performance in distinguishing between **LOS** and **NLOS**. However, reducing **False Negatives** could further enhance its ability to identify **NLOS** instances.

To further assess the model's performance, we evaluate the **ROC-AUC** scores. The **Train ROC-AUC** score of **0.9185** and the **Test ROC-AUC** score of **0.9187** indicate that the Logistic Regression model performs consistently well across both the training and test data. These scores suggest strong generalization, as the values are almost identical, showing that the model is not overfitting.

The **Classification Report** provides further insights into the model's performance. The precision and recall for both classes are quite balanced, with **precision** for **LOS (0.0)** at **0.82** and **NLOS (1.0)** at **0.90**. The **recall** for **LOS (0.0)** is **0.91**, indicating good identification of **LOS** instances, while the **recall** for **NLOS (1.0)** is **0.80**, showing that the model misses a few **NLOS** instances. However, the **f1-scores** for both classes are **0.86**, indicating overall good performance for both classes.

Overall, the **ROC-AUC** values show that the model is effective at distinguishing between **LOS** and **NLOS**. The **classification report** supports this by showing a good balance in performance, especially in terms of precision, recall, and f1-scores.

In [ ]:
# --- Logistic Regression Model Evaluation ---
# Predict using the best Logistic Regression model
y_pred_lr = best_lr_model.predict(X_test_scaled)

# Calculate accuracy of the Logistic Regression model
accuracy_lr = accuracy_score(y_test, y_pred_lr)

# Generate the confusion matrix for the Logistic Regression model
conf_matrix_lr = confusion_matrix(y_test, y_pred_lr)

# Extract true positives (TP), true negatives (TN), false positives (FP), and false negatives (FN)
TP_lr, TN_lr, FP_lr, FN_lr = (
    conf_matrix_lr[1, 1],
    conf_matrix_lr[0, 0],
    conf_matrix_lr[0, 1],
    conf_matrix_lr[1, 0],
)

# Print out the evaluation metrics for the Logistic Regression model
print(f"Logistic Regression Accuracy: {accuracy_lr:.4f}")
print(f"Logistic Regression Confusion Matrix:\n{conf_matrix_lr}")
print(
    f"Logistic Regression Classification Report:\n{classification_report(y_test, y_pred_lr)}"
)

###### **3.1.3 - visualization graph* (to be added)* 

##### **3.1.1.4 - Evaluation and Performance of the Best Logistic Regression Model (description to be modify)**


After performing hyperparameter tuning, the model was evaluated to assess its performance. The evaluation focused on key metrics such as accuracy, precision, recall, and ROC-AUC on the test dataset, to determine the model’s ability to generalize to unseen data.


The optimized Logistic Regression model achieved an accuracy of 85.60%. It performed well in identifying NLOS, with 3871 True Negatives, but exhibited a higher False Negative rate for LOS (826).  Precision was higher for LOS (0.90), while recall was better for NLOS (0.91), suggesting a slight imbalance in the model's predictions.

To further evaluate the model, we make predictions on the test data and calculate key metrics such as accuracy and confusion matrix:


In [ ]:
y_pred_best_lr = best_lr_model.predict(X_test)
accuracy_best_lr = accuracy_score(y_test, y_pred_best_lr)
conf_matrix_best_lr = confusion_matrix(y_test, y_pred_best_lr)

y_train_pred_best_lr = best_lr_model.predict_proba(X_train)[:, 1]
y_test_pred_best_lr = best_lr_model.predict_proba(X_test)[:, 1]

train_auc_best_lr = roc_auc_score(y_train, y_train_pred_best_lr)
test_auc_best_lr = roc_auc_score(y_test, y_test_pred_best_lr)


print(f"Best Logistic Regression Accuracy: {accuracy_best_lr:.4f}")
print(f"Best Logistic Regression Confusion Matrix:\n{conf_matrix_best_lr}")
print(
    f"Best Logistic Regression Classification Report:\n{classification_report(y_test, y_pred_best_lr)}"
)

print(
    f"Best Logistic Regression - Train ROC-AUC: {train_auc_best_lr:.4f}, Test ROC-AUC: {test_auc_best_lr:.4f}"
)

###### **3.1.3 - visualization graph** 

**(description to be modify)** 

The ROC curve below visualizes the performance of the Best Logistic Regression model. With an AUC of 0.9187, the model demonstrates a strong ability to distinguish between LOS and NLOS classes. The curve significantly exceeds the random guessing baseline (the gray dashed line), indicating the model's high accuracy. The close proximity of the model's ROC AUC to 1.0 suggests that the model is highly effective at identifying both classes. However, there's always potential to fine-tune the model to improve performance further.


To further assess the model's performance, we evaluate key metrics, including **accuracy, confusion matrix, and classification report**. The **Logistic Regression** model achieved an **accuracy of 0.8560**, indicating strong overall performance across both the training and test data.  

The **Confusion Matrix** provides further insights into the model's classification performance. The model correctly identified **3870 True Negatives** and **3320 True Positives**, while misclassifying **382 False Positives** and **828 False Negatives**. These values suggest that the model performs well but misses some **NLOS** instances.  

The **Classification Report** provides additional details on precision, recall, and f1-score. The **precision** for **LOS (0.0)** is **0.82**, while for **NLOS (1.0)** it is **0.90**, indicating that the model is slightly better at predicting **NLOS** instances correctly. The **recall** for **LOS (0.0)** is **0.91**, demonstrating strong identification of **LOS**, whereas the **recall** for **NLOS (1.0)** is **0.80**, meaning some **NLOS** cases are misclassified. The **f1-scores** for both classes are balanced, with **LOS (0.0) at 0.86** and **NLOS (1.0) at 0.85**, reflecting the model’s overall effectiveness.  

Overall, the model shows strong performance in distinguishing between **LOS** and **NLOS**. However, reducing **False Negatives** could further enhance its ability to identify **NLOS** instances.

In [ ]:
fpr_train, tpr_train, _ = roc_curve(y_train, y_train_pred_best_lr)
fpr_test, tpr_test, _ = roc_curve(y_test, y_test_pred_best_lr)

train_auc = auc(fpr_train, tpr_train)
test_auc = auc(fpr_test, tpr_test)

plt.figure(figsize=(8, 6))
plt.plot(fpr_train, tpr_train, color="blue", lw=2, label=f"Train ROC curve (AUC = {train_auc:.2f})")
plt.plot(fpr_test, tpr_test, color="red", lw=2, label=f"Test ROC curve (AUC = {test_auc:.2f})")
plt.plot([0, 1], [0, 1], color="gray", linestyle="--")  # Diagonal line for random classifier
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve for Best Logistic Regression Model")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


##### **3.1.1.5 - Confusion Matrix Visualization (description to be added)**

XXX

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(conf_matrix_best_lr, interpolation="nearest", cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.colorbar()
tick_marks = [0, 1]
plt.xticks(tick_marks, tick_marks)
plt.yticks(tick_marks, tick_marks)

thresh = conf_matrix_best_lr.max() / 2.  # Threshold to make text color readable
for i, j in itertools.product(range(conf_matrix_best_lr.shape[0]), range(conf_matrix_best_lr.shape[1])):
    plt.text(j, i, format(conf_matrix_best_lr[i, j], 'd'),
             horizontalalignment="center",
             color="white" if conf_matrix_best_lr[i, j] > thresh else "black")
    
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()


##### **3.1.1.6 - Conclusion: Performance of the Best Logistic Regression Mode (to be modify)**

In conclusion, the **best Logistic Regression model** achieved an accuracy of 85.63%, demonstrating solid performance in classifying both NLOS and LOS instances.

The confusion matrix shows the model is effective at identifying NLOS (3871 True Negatives) but has a higher False Negative rate for LOS (826). The classification report indicates high precision for LOS (0.90) and strong recall for NLOS (0.91), with an F1-score of 0.87 for NLOS and 0.85 for LOS. 

The ROC-AUC scores (Train: 0.9185, Test: 0.9187) confirm the model's robust ability to distinguish between the two classes. Overall, the model is reliable, with a slight imbalance favoring the correct identification of NLOS instances.

#### **3.1.2 - Support Vector Machine (SVM)**

First, the SVM model is initialized with probability output and trained on the training data to classify the target labels.

In [ ]:
# import libaries
try:
    from sklearn.svm import SVC

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

In [ ]:
svm_model = SVC(probability=True, random_state=42)
svm_model.fit(X_train, y_train)

##### **3.1.2.1 - Hyperparameter Tuning** 

To further enhance the model's performance, we perform hyperparameter tunning to  optimize the performance of the SVM model by selecting the best combination of parameters. In this case, GridSearchCV was used to tune the C, gamma, and kernel hyperparameters.

In [ ]:
param_grid = {
    "C": [0.1, 1, 10],  # Regularization parameter
    "kernel": ["linear"],  # Linear kernel
}

grid_search = GridSearchCV(
    svm_model,
    param_grid=param_grid,
    cv=2,  # 2-fold cross-validation
    n_jobs=-1,
    verbose=1,  # Reduced verbosity
    scoring="roc_auc",  # Evaluation based on ROC-AUC
)

grid_search.fit(X_train, y_train)

# Print the best hyperparameters found by GridSearchCV
print(f"Best hyperparameters found: {grid_search.best_params_}")

##### **3.1.2.2 - Model Training and Testing (Description to be modify)** 

**Training and Test ROC-AUC for SVM**


To further assess the model’s ability to distinguish between the **NLOS** and **LOS** classes, we evaluate the **ROC-AUC** scores. The **Train ROC-AUC** score of **0.9212** and the **Test ROC-AUC** score of **0.9216** indicate that the model performs consistently well across both the training and test data. These high scores suggest that the SVM model has strong generalization and a robust ability to classify the data correctly. A **ROC-AUC** value close to 1 indicates the model's ability to distinguish between the classes with high accuracy, confirming the model’s effectiveness.

Additionally, the **Confusion Matrix** provides further insights into the classification performance. It shows that the model correctly classified a high number of **True Positives (3188)** and **True Negatives (3954)**, while the **False Positives (298)** and **False Negatives (960)** are relatively low. This highlights that the model has a good balance between precision and recall, performing well in identifying both classes.

The **Classification Report** reinforces these findings by providing metrics such as **precision**, **recall**, and **f1-score**. The precision for **LOS (0.0)** is **0.80**, indicating that 80% of predicted **LOS** instances are correct, while the precision for **NLOS (1.0)** is **0.91**, showing that the model is more accurate in predicting **NLOS** instances. The **recall** values of **0.93** for **LOS** and **0.77** for **NLOS** show that the model is better at identifying **LOS** instances, but still performs well for **NLOS**. The **f1-score**, which balances both precision and recall, is **0.86** for both classes, showing an overall good classification performance.

These results together demonstrate that the SVM model is both effective and reliable in distinguishing between **NLOS** and **LOS** classes, with minimal overfitting and strong generalization.

In [ ]:
# --- SVM ROC-AUC Evaluation for Train and Test ---
# Use the best model from grid search
best_svm_model = grid_search.best_estimator_

# Predict probabilities for the positive class (class 1) for training and test data
y_train_pred_best_svm = best_svm_model.predict_proba(X_train)[:, 1]
y_test_pred_best_svm = best_svm_model.predict_proba(X_test)[:, 1]

# Compute ROC-AUC scores for both train and test sets
train_auc_best_svm = roc_auc_score(y_train, y_train_pred_best_svm)
test_auc_best_svm = roc_auc_score(y_test, y_test_pred_best_svm)

# Print AUC for train and test for SVM
print(
    f"Best SVM - Train ROC-AUC: {train_auc_best_svm:.4f}, Test ROC-AUC: {test_auc_best_svm:.4f}"
)

###### **3.1.3 - visualization graph** 

In [ ]:
# Predict probabilities for SVM on train and test data
y_train_pred_best_svm = best_svm_model.predict_proba(X_train)[:, 1]
y_test_pred_best_svm = best_svm_model.predict_proba(X_test)[:, 1]

# ROC curve for SVM (train and test)
fpr_train_svm, tpr_train_svm, _ = roc_curve(y_train, y_train_pred_best_svm)
fpr_test_svm, tpr_test_svm, _ = roc_curve(y_test, y_test_pred_best_svm)

# Compute AUC for both train and test
train_auc_svm = auc(fpr_train_svm, tpr_train_svm)
test_auc_svm = auc(fpr_test_svm, tpr_test_svm)

# Plot ROC curves for SVM model
plt.figure(figsize=(8, 6))
plt.plot(
    fpr_train_svm,
    tpr_train_svm,
    color="blue",
    label="Train ROC curve (AUC = {:.2f})".format(train_auc_svm),
)
plt.plot(
    fpr_test_svm,
    tpr_test_svm,
    color="red",
    label="Test ROC curve (AUC = {:.2f})".format(test_auc_svm),
)
plt.plot([0, 1], [0, 1], color="gray", linestyle="--")  # Random classifier line
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Best SVM Model")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

##### **3.1.2.3 - Model Evaluation (description to be modify)**

After training the SVM model, we evaluate its performance on the test data by calculating key metrics such as accuracy, confusion matrix, and classification report. These metrics provide valuable insights into the model's effectiveness in classifying **LOS** and **NLOS** instances.

The model achieved an accuracy of **85.02%**. It showed stronger performance in identifying **NLOS** instances, with precision and recall for **NLOS** being 1.0. In contrast, the model struggled more with correctly identifying **LOS**, as indicated by its 0.0 precision for this class. The confusion matrix further reveals that the model has a relatively low number of false positives and false negatives, suggesting generally solid classification performance. However, the disparity in precision and recall indicates that there is room for improvement, especially in correctly classifying **LOS** instances.

In [ ]:
# --- SVM Model Evaluation ---
# Predict using the best SVM model
y_pred_best_svm = best_svm_model.predict(X_test)  # Prediction for test set

# Calculate accuracy of the SVM model
accuracy_best_svm = accuracy_score(y_test, y_pred_best_svm)

# Generate the confusion matrix for the SVM model
conf_matrix_best_svm = confusion_matrix(y_test, y_pred_best_svm)  # Confusion Matrix

# Extract true positives (TP), true negatives (TN), false positives (FP), and false negatives (FN)
TP_best_svm, TN_best_svm, FP_best_svm, FN_best_svm = (
    conf_matrix_best_svm[1, 1],
    conf_matrix_best_svm[0, 0],
    conf_matrix_best_svm[0, 1],
    conf_matrix_best_svm[1, 0],
)

# Print out the evaluation metrics for the SVM model
print(f"Best SVM Accuracy: {accuracy_best_svm:.4f}")
print(f"Best SVM Confusion Matrix:\n{conf_matrix_best_svm}")
print(
    f"Best SVM Classification Report:\n{classification_report(y_test, y_pred_best_svm)}"
)

###### **3.1.3 - visualization graph** 

In [ ]:
# ROC curve for train and test sets
plt.figure(figsize=(8, 6))
plt.plot(
    fpr_train,
    tpr_train,
    color="blue",
    label=f"Train ROC curve (AUC = {train_auc_svm:.4f})",
)
plt.plot(
    fpr_test, tpr_test, color="red", label=f"Test ROC curve (AUC = {test_auc_svm:.4f})"
)
plt.plot([0, 1], [0, 1], color="gray", linestyle="--")  # Random classifier line
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Best SVM Model")
plt.legend(loc="lower right")
plt.show()

After performing hyperparameter tuning, the model was evaluated to assess its performance. The evaluation focused on key metrics such as accuracy, precision, recall, and ROC-AUC on the test dataset, to determine the model’s ability to generalize to unseen data.

##### **3.1.2.4 - Evaluation and Performance of the Best SVM Model (description to be modify)**

In [ ]:
# Best SVM - Accuracy, Confusion Matrix, and Classification Report
print("\n### SVM Evaluation ###")
print(f"Best SVM Accuracy: {accuracy_best_svm:.4f}")
print(f"Best SVM Confusion Matrix:\n{conf_matrix_best_svm}")
print(
    f"Best SVM Classification Report:\n{classification_report(y_test, y_pred_best_svm)}"
)

# Print AUC for both models
train_auc_svm = roc_auc_score(y_train, y_train_pred_best_svm)
test_auc_svm = roc_auc_score(y_test, y_test_pred_best_svm)
print(
    f"Best SVM - Train ROC-AUC: {train_auc_svm:.4f}, Test ROC-AUC: {test_auc_svm:.4f}"
)

###### **3.1.3 - visualization graph** 

In [ ]:
# Predict probabilities for train and test sets
y_train_pred_best_svm = best_svm_model.predict_proba(X_train)[
    :, 1
]  # Predict probabilities for class 1
y_test_pred_best_svm = best_svm_model.predict_proba(X_test)[
    :, 1
]  # Predict probabilities for class 1

# Compute ROC curve for train and test sets
fpr_train, tpr_train, _ = roc_curve(y_train, y_train_pred_best_svm)
fpr_test, tpr_test, _ = roc_curve(y_test, y_test_pred_best_svm)

# Compute AUC for train and test
train_roc_auc = auc(fpr_train, tpr_train)
test_roc_auc = auc(fpr_test, tpr_test)

# Plot ROC curves for train and test sets
plt.figure(figsize=(8, 6))
plt.plot(
    fpr_train,
    tpr_train,
    color="blue",
    label=f"Train ROC curve (AUC = {train_roc_auc:.4f})",
)
plt.plot(
    fpr_test, tpr_test, color="red", label=f"Test ROC curve (AUC = {test_roc_auc:.4f})"
)
plt.plot([0, 1], [0, 1], color="gray", linestyle="--")  # Random classifier line
plt.title("ROC Curve - Best SVM Model")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

##### **3.1.1.6 - Confusion Matrix Visualization (description to be added)**

In [ ]:
# Confusion matrix plot
plt.figure(figsize=(8, 6))
plt.imshow(conf_matrix_best_svm, interpolation="nearest", cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.colorbar()
tick_marks = [0, 1]
plt.xticks(tick_marks, tick_marks)
plt.yticks(tick_marks, tick_marks)

thresh = conf_matrix_best_svm.max() / 2.  # Threshold to make text color readable
for i, j in itertools.product(range(conf_matrix_best_svm.shape[0]), range(conf_matrix_best_svm.shape[1])):
    plt.text(j, i, format(conf_matrix_best_svm[i, j], 'd'),
             horizontalalignment="center",
             color="white" if conf_matrix_best_svm[i, j] > thresh else "black")

plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()


##### **3.1.2.5 - Conclusion: Performance of the Best SVM Model (Description to be added)**

============= FOLLOW THIS FORMAT!! =============
    
In conclusion, the **best Logistic SVM model** achieved an accuracy of 85.63%, demonstrating solid performance in classifying both NLOS and LOS instances.

The confusion matrix shows the model is effective at identifying NLOS (3871 True Negatives) but has a higher False Negative rate for LOS (826). The classification report indicates high precision for LOS (0.90) and strong recall for NLOS (0.91), with an F1-score of 0.87 for NLOS and 0.85 for LOS. 

The ROC-AUC scores (Train: 0.9185, Test: 0.9187) confirm the model's robust ability to distinguish between the two classes. Overall, the model is reliable, with a slight imbalance favoring the correct identification of NLOS instances.

#### **3.1.3 - Random Forest**

##### **3.1.3.3 - Hyperparameter Tuning (Description to be added)** 

XXXX

In [ ]:
# --- Hyperparameter Tuning with GridSearchCV (if applicable) ---
param_grid_rf = {
    "n_estimators": [100, 200, 300],  # Number of trees
    "max_depth": [None, 10, 20, 30],  # Maximum depth of the tree
    "min_samples_split": [
        2,
        5,
        10,
    ],  # Minimum number of samples required to split a node
}

grid_search_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid_rf,
    cv=3,  # 3-fold cross-validation
    n_jobs=-1,
    verbose=1,  # Display progress
    scoring="roc_auc",  # Optimize for ROC-AUC
)

# Fit the grid search
grid_search_rf.fit(X_train, y_train)

# Print best hyperparameters
print(f"Best Random Forest hyperparameters: {grid_search_rf.best_params_}")

##### **3.1.3.2 - Model Training and Testing (Description to be modify)** 

XXX

In [ ]:
# --- ROC-AUC Calculation (Train and Test) ---
# Use the best estimator from grid search
best_rf_model = grid_search_rf.best_estimator_

# Predict probabilities for train and test sets using the best model
y_train_pred_rf = best_rf_model.predict_proba(X_train)[:, 1]
y_test_pred_rf = best_rf_model.predict_proba(X_test)[:, 1]

# Compute ROC-AUC for train and test
train_auc_rf = roc_auc_score(y_train, y_train_pred_rf)
test_auc_rf = roc_auc_score(y_test, y_test_pred_rf)

# Print ROC-AUC scores
print(
    f"Random Forest - Train ROC-AUC: {train_auc_rf:.4f}, Test ROC-AUC: {test_auc_rf:.4f}"
)

###### **3.1.3 - visualization graph** 

In [ ]:
# Predict probabilities for SVM on train and test data
y_train_pred_best_svm = best_svm_model.predict_proba(X_train)[:, 1]
y_test_pred_best_svm = best_svm_model.predict_proba(X_test)[:, 1]

# ROC curve for SVM (train and test)
fpr_train_svm, tpr_train_svm, _ = roc_curve(y_train, y_train_pred_best_svm)
fpr_test_svm, tpr_test_svm, _ = roc_curve(y_test, y_test_pred_best_svm)

# Compute AUC for both train and test
train_auc_svm = auc(fpr_train_svm, tpr_train_svm)
test_auc_svm = auc(fpr_test_svm, tpr_test_svm)

# Plot ROC curves for SVM model
plt.figure(figsize=(8, 6))
plt.plot(
    fpr_train_svm,
    tpr_train_svm,
    color="blue",
    label="Train ROC curve (AUC = {:.2f})".format(train_auc_svm),
)
plt.plot(
    fpr_test_svm,
    tpr_test_svm,
    color="red",
    label="Test ROC curve (AUC = {:.2f})".format(test_auc_svm),
)
plt.plot([0, 1], [0, 1], color="gray", linestyle="--")  # Random classifier line
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Best SVM Model")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

##### **3.1.3.1 - Model Evaluation**

In [ ]:
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)  # Train

In [ ]:
# --- Model Evaluation ---
y_pred_rf = rf_model.predict(X_test)  # Prediction on test set

# Calculate accuracy
accuracy_rf = accuracy_score(y_test, y_pred_rf)

# Generate confusion matrix
conf_matrix_rf = confusion_matrix(y_test, y_pred_rf)

# First 10 Actual vs Predicted values
comparison_rf = pd.DataFrame(
    {"Actual": y_test[:10].values, "Predicted": y_pred_rf[:10]}
)

# Print evaluation results
print(f"Random Forest Accuracy: {accuracy_rf:.4f}")
print(f"Random Forest Confusion Matrix:\n{conf_matrix_rf}")
print(
    f"Random Forest Classification Report:\n{classification_report(y_test, y_pred_rf)}"
)
print("\nFirst 10 Predicted vs. Actual Values (Random Forest):\n", comparison_rf)

###### **3.1.3 - visualization graph** 

In [ ]:
fpr_train_lr, tpr_train_lr, _ = roc_curve(y_train, y_train_pred_lr)
fpr_test_lr, tpr_test_lr, _ = roc_curve(y_test, y_test_pred_lr)

plt.figure(figsize=(8, 6))
plt.plot(
    fpr_train_lr,
    tpr_train_lr,
    color="blue",
    label="Train ROC curve (AUC = {:.2f})".format(train_auc_lr),
)
plt.plot(
    fpr_test_lr,
    tpr_test_lr,
    color="red",
    label="Test ROC curve (AUC = {:.2f})".format(test_auc_lr),
)
plt.plot([0, 1], [0, 1], color="gray", linestyle="--")  # Random classifier line
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend(loc="lower right")
plt.grid(True)

##### **3.1.2.4 - Evaluation and Performance of the Best Rnadom Forest Model (need to add???)**

In [ ]:
# --- Model Evaluation ---
y_pred_rf = rf_model.predict(X_test)  # Prediction on test set

# Calculate accuracy
accuracy_rf = accuracy_score(y_test, y_pred_rf)

# Generate confusion matrix
conf_matrix_rf = confusion_matrix(y_test, y_pred_rf)

# First 10 Actual vs Predicted values
comparison_rf = pd.DataFrame(
    {"Actual": y_test[:10].values, "Predicted": y_pred_rf[:10]}
)

# Print evaluation results
print(f"Random Forest Accuracy: {accuracy_rf:.4f}")
print(f"Random Forest Confusion Matrix:\n{conf_matrix_rf}")
print(f"Random Forest Classification Report:\n{classification_report(y_test, y_pred_rf)}")
print("\nFirst 10 Predicted vs. Actual Values (Random Forest):\n", comparison_rf)


###### **3.1.3 - visualization graph** 

In [ ]:
fpr_train_lr, tpr_train_lr, _ = roc_curve(y_train, y_train_pred_lr)
fpr_test_lr, tpr_test_lr, _ = roc_curve(y_test, y_test_pred_lr)

plt.figure(figsize=(8, 6))
plt.plot(
    fpr_train_lr,
    tpr_train_lr,
    color="blue",
    label="Train ROC curve (AUC = {:.2f})".format(train_auc_lr),
)
plt.plot(
    fpr_test_lr,
    tpr_test_lr,
    color="red",
    label="Test ROC curve (AUC = {:.2f})".format(test_auc_lr),
)
plt.plot([0, 1], [0, 1], color="gray", linestyle="--")  # Random classifier line
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend(loc="lower right")
plt.grid(True)

##### **3.1.3.6 - Conclusion: Performance of the Best Random Forest Model (descrioption to be added)**

XXX 

In [ ]:
# --- Confusion Matrix Visualization ---
plt.figure(figsize=(8, 6))
plt.imshow(conf_matrix_rf, interpolation="nearest", cmap=plt.cm.Blues)
plt.title("Random Forest Confusion Matrix")
plt.colorbar()

# Add tick marks and labels
tick_marks = [0, 1]
plt.xticks(tick_marks, tick_marks)
plt.yticks(tick_marks, tick_marks)

# Add text inside the confusion matrix boxes
thresh = conf_matrix_rf.max() / 2.0  # Threshold for text color
for i, j in itertools.product(
    range(conf_matrix_rf.shape[0]), range(conf_matrix_rf.shape[1])
):
    plt.text(
        j,
        i,
        format(conf_matrix_rf[i, j], "d"),
        horizontalalignment="center",
        color="white" if conf_matrix_rf[i, j] > thresh else "black",
    )

# Add labels and layout adjustments
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()

## **4 - Model Comparison and Analysis**

After training and tuning the models for Random Forest, SVM, and Logistic Regression, we evaluate their performance using key metrics and visualizations. These evaluations allow us to compare the models and identify the most suitable one for the task.

##### **4.1 - Model Evaluation**


After obtaining the best models for Random Forest, SVM, and Logistic Regression, we assess their performance on the test set using several key metrics and visualizations:

- **Accuracy**: Measures the proportion of correctly predicted instances.
- **AUC (Area Under the Curve)**: Provides a performance evaluation of the model, particularly useful for understanding the trade-off between True Positive Rate and False Positive Rate.
- **Confusion Matrix**: Shows a detailed breakdown of the classification results, including True Positives, False Positives, True Negatives, and False Negatives.
- **Precision-Recall Curve**: Visualizes the trade-off between Precision and Recall, especially important for imbalanced datasets.

By comparing these metrics and visualizations, we can determine the best-performing model and select the most appropriate one for the given problem. The selected model is then saved for future use.

In [ ]:
# random forest
rf_accuracy = accuracy_score(y_test, best_rf_model.predict(X_test))
rf_auc = roc_auc_score(y_test, best_rf_model.predict_proba(X_test)[:, 1])

# Evaluate SVM
svm_accuracy = accuracy_score(y_test, best_svm_model.predict(X_test))
svm_auc = roc_auc_score(y_test, best_svm_model.predict_proba(X_test)[:, 1])

# Evaluate Logistic Regression
lr_accuracy = accuracy_score(y_test, best_lr_model.predict(X_test))
lr_auc = roc_auc_score(y_test, best_lr_model.predict_proba(X_test)[:, 1])

print(f"Random Forest AUC: {rf_auc:.4f}, Accuracy: {rf_accuracy:.4f}")
print(f"SVM AUC: {svm_auc:.4f}, Accuracy: {svm_accuracy:.4f}")
print(f"Logistic Regression AUC: {lr_auc:.4f}, Accuracy: {lr_accuracy:.4f}")

In [ ]:
# Metrics for plotting
models = ["Random Forest", "SVM", "Logistic Regression"]
auc_scores = [rf_auc, svm_auc, lr_auc]
accuracy_scores = [rf_accuracy, svm_accuracy, lr_accuracy]

# Plotting AUC and Accuracy
x = np.arange(len(models))  # The label locations
width = 0.35  # The width of the bars

fig, ax = plt.subplots(figsize=(8, 6))

bars1 = ax.bar(x - width / 2, auc_scores, width, label="AUC", color="skyblue")
bars2 = ax.bar(x + width / 2, accuracy_scores, width, label="Accuracy", color="orange")

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_xlabel("Models")
ax.set_ylabel("Scores")
ax.set_title("Model Performance Comparison (AUC and Accuracy)")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()


# Attach the text labels above the bars
def add_text(bars, values):
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax.annotate(
            f"{value:.4f}",  # Display the value with 4 decimal places
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),  # 3 points vertical offset
            textcoords="offset points",
            ha="center",
            va="bottom",
        )


add_text(bars1, auc_scores)
add_text(bars2, accuracy_scores)

plt.tight_layout()
plt.show()

**Model Comparison and Selection of the Best Model**

After training and evaluating the models for Random Forest, SVM, and Logistic Regression, we compare their performance based on several key metrics. These metrics include Accuracy, AUC, Confusion Matrix, and Precision-Recall Curve. By analyzing these evaluations, we can determine which model best meets the requirements of the task at hand.

The model that performs the best across these evaluations is selected as the final model. This model is then saved for future use, ensuring that it can be deployed or further fine-tuned as needed.

In [ ]:
if rf_auc > svm_auc and rf_auc > lr_auc:
    best_model = best_rf_model
    print("Best Model: Random Forest")
elif svm_auc > rf_auc and svm_auc > lr_auc:
    best_model = best_svm_model
    print("Best Model: SVM")
else:
    best_model = best_lr_model
    print("Best Model: Logistic Regression")

##### **4.2 - ROC Curve Analysis of Model Performance**

In [ ]:
# ROC Curve for the best model
fpr, tpr, _ = roc_curve(y_test, best_model.predict_proba(X_test)[:, 1])

# Plotting the ROC curve
plt.figure(figsize=(6, 5))
plt.plot(
    fpr, tpr, color="b", label="ROC Curve"
)  # Plot the ROC curve for the best model
plt.plot(
    [0, 1], [0, 1], color="r", linestyle="--", label="Random Classifier"
)  # Random classifier (diagonal)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Test Set ROC Curve")
plt.legend(loc="lower right")
plt.show()

## **5 - Model Deployment**
After selecting and evaluating the best model, we save the trained model for future use and easy deployment. This allows us to utilize the model without needing to retrain it each time. The saved model can be loaded into production environments for making predictions on new, unseen data.

In [ ]:
# import libraries

try:
    import os
    import joblib
    from sklearn.metrics import roc_auc_score

    print("Libraries imported successfully!")
except ImportError as e:
    print(f"Error importing libraries: {e}")

### **5.1 - Create Directories for Storing Models**

In [ ]:
# Create the main directory for final models if it doesn't exist
folder_name = "final_models"
os.makedirs(folder_name, exist_ok=True)

# Create a subdirectory specifically for algorithm models
algo_folder = os.path.join(folder_name, "algo_models")
os.makedirs(algo_folder, exist_ok=True)

### **5.2 - Save the Best Overall Model**

In [ ]:
# Save the overall best model (across all algorithms)
overallBest_filename = os.path.join(folder_name, "best_overall_model.pkl")
joblib.dump(best_model, overallBest_filename)
print(f"Final model saved as {overallBest_filename}")

### **5.3 - Save the Best Models for Each Algorithm**

#### **5.3.1 - Logistic Regression Model**

In [ ]:
# Save the best Logistic Regression model
LR_filename = os.path.join(algo_folder, "logistic_regression_model.pkl")
joblib.dump(best_lr_model, LR_filename)
print(f"Logistic Regression model saved as {LR_filename}")

#### **5.3.2 - SVM Model**

In [ ]:
# Save the best SVM model
SVM_filename = os.path.join(algo_folder, "SVM_model.pkl")
joblib.dump(best_svm_model, SVM_filename)
print(f"SVM model saved as {SVM_filename}")

#### **5.3.3 - Random Forest Model**

In [ ]:
# Save the best Random Forest model
RF_filename = os.path.join(algo_folder, "RF_model.pkl")
joblib.dump(best_rf_model, RF_filename)
print(f"Random Forest model saved as {RF_filename}")